
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F3EB8B4ED0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F3EBA1E850>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from pydantic import Field, BaseModel

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: int=Field(description="The year the movie was released")
    director: str=Field(description="The director of the movie")
    rating: float=Field(description="The movies rating out of 10")

In [5]:
model_with_structured = model.with_structured_output(Movie)
model_with_structured

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F3EB8B4ED0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F3EBA1E850>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'd

In [6]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='<think>\nOkay, so I need to figure out details about the movie Inception. Let me start by recalling what I know. Inception is a sci-fi action movie directed by Christopher Nolan. The main actor is Leonardo DiCaprio. The plot involves something about dreams and planting ideas. I remember there\'s a concept where people can enter each other\'s dreams to steal or plant ideas, which is called "inception." The movie has a lot of layers and twists, and the ending is pretty famous for the spinning top. \n\nWait, what\'s the main character\'s name? I think he\'s called Dom Cobb, played by DiCaprio. His character has a family issue; he has a wife who died or something? Maybe she\'s linked to the idea of dreams. There\'s a team he assembles for the heist. The mission is to plant an idea into someone\'s mind, right? The target is a businessman named Robert Fischer. The team includes someone who\'s a dream architect, a forger, a man who can navigate dreams, and maybe a driver or

In [8]:
response = model_with_structured.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [9]:
from pydantic import Field, BaseModel

class Movie(BaseModel):
    """A movie with details"""
    title: str=Field(..., description="The title of the movie")
    year: int=Field(..., description="The year the movie was released")
    director: str=Field(..., description="The director of the movie")
    rating: float=Field(..., description="The movies rating out of 10")

model_with_structured = model.with_structured_output(Movie, include_raw=True)

response = model_with_structured.invoke("Provide details about the moview Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what functions I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all that information for Inception.\n\nFirst, the title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb lists it at 8.8/10. \n\nSo all the required parameters are present. I should structure the tool call with these details. Make sure the JSON is correctly formatted with the right data types: title and director as strings, year as an integer, and rating as a number. No need for any optional parameters here since all required fields are covered. Alright, that should do it.\n', 'tool_calls': [{'id': 'vhmp443nc', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8

### Nested Structure

In [10]:
from pydantic import Field, BaseModel

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="The budget in millions USD")

model_with_structured = model.with_structured_output(MovieDetails)
response = model_with_structured.invoke("Provide details about the moview Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Jason')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [11]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    title: Annotated[str, ... ,"The title of the movie"]
    year: Annotated[int, ... ,"The year the movie was released"]
    director: Annotated[str, ... ,"The director of the movie"]
    rating: Annotated[float, ... ,"The movies rating out of 10"]


model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("Provide details about the moview Avengers")
response

{'director': 'Joss Whedon', 'rating': 8.1, 'title': 'Avengers', 'year': 2012}

In [12]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="The budget in millions USD")

model_with_typedict = model.with_structured_output(MovieDetails)
response = model_with_typedict.invoke("Provide details about the moview Avengers")
response
# OUTPUT:

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'genres': ['Action', 'Science Fiction'],
 'title': 'Avengers',
 'year': 2012}

In [13]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [15]:
# import os
# os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [18]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact informatin of a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(model="groq:qwen/qwen3-32b",
                     response_format=ContactInfo
                     )


result = agent.invoke({"messages": [{"role": "user", "content": "Provide contact information for John Doe, jhon@example.com, (555) 123-4567"}]})
print(result["structured_response"])
result

name='John Doe' email='jhon@example.com' phone='(555) 123-4567'


{'messages': [HumanMessage(content='Provide contact information for John Doe, jhon@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='66645260-0c40-402d-ae5b-07cb906fa7df'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for contact information for John Doe. Let me check the tools provided. There's a function called ContactInfo that requires name, email, and phone. The user provided all three: name is John Doe, email is jhon@example.com, and phone is (555) 123-4567. I need to make sure the parameters match exactly. The email has a typo, 'jhon' instead of 'john', but I should use it as provided. The phone number includes parentheses and a space, but the function just takes it as a string. So I'll format the JSON with those values. Let me double-check the required fields. All three are required, so that's covered. Alright, the tool call should be correct.\n", 'tool_calls': [{'id': '1dg4adev7', 'function': {'arguments':

In [19]:
result["structured_response"]

ContactInfo(name='John Doe', email='jhon@example.com', phone='(555) 123-4567')

In [21]:
from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact informatin of a person"""
    name: str 
    email: str 
    phone: str 


agent = create_agent(model="groq:qwen/qwen3-32b",
                     response_format=ContactInfo
                     )


result = agent.invoke({"messages": [{"role": "user", "content": "Provide contact information for John Doe, jhon@example.com, (555) 123-4567"}]})
print(result["structured_response"])
result

{'name': 'John Doe', 'email': 'jhon@example.com', 'phone': '(555) 123-4567'}


{'messages': [HumanMessage(content='Provide contact information for John Doe, jhon@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='2711e9f3-6acb-45a6-9e58-a5953b7e0cc8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for contact information for John Doe. Let me check the tools provided. There's a function called ContactInfo that requires name, email, and phone. The user provided all three: name is John Doe, email is jhon@example.com, and phone is (555) 123-4567. I need to make sure the parameters match exactly. The email has a typo, 'jhon' instead of 'john', but I should use it as given. The phone number includes parentheses and a space, but the function doesn't specify formatting, so I'll input it as provided. Alright, I'll structure the JSON with those details.\n", 'tool_calls': [{'id': 'cpwyapadd', 'function': {'arguments': '{"email":"jhon@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'Con

In [22]:
## dataclass


from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact informatin of a person"""
    name: str 
    email: str 
    phone: str

agent = create_agent(model="groq:qwen/qwen3-32b",
                     response_format=ContactInfo
                     )

result = agent.invoke({"messages": [{"role": "user", "content": "Provide contact information for John Doe, jhon@example.com, (555) 123-4567"}]})
print(result["structured_response"])
# result


ContactInfo(name='John Doe', email='jhon@example.com', phone='(555) 123-4567')
